# Logistic Regression - Probabilities, Decision Boundaries, and Pipelines

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/06_logistic_pipelines_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Fit logistic regression with preprocessing in a pipeline
2. Interpret probabilities vs classes (and why thresholds matter)
3. Use regularization in logistic regression for stability
4. Choose an appropriate baseline for classification
5. Document the classification objective and error costs

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete all exercises before submitting your notebook.

---

## 💼 Why This Matters: From Prices to Probabilities

A new client arrives at your desk: the **State Health Department** wants a breast cancer screening tool. They have digitized cell measurements from fine-needle aspirates — 30 numeric features describing each cell nucleus (radius, texture, perimeter, area, smoothness...). The question is no longer "how much?" but "which category?" — malignant or benign.

This shift from predicting a number to predicting a class changes everything: the model, the metrics, the pipeline, and the stakes. A wrong house price estimate costs dollars; a wrong cancer diagnosis can cost a life.

> **Today's focus:** Building our first classification model with logistic regression, and constructing pipelines that handle the breast cancer dataset from raw features to probability predictions.

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
print("✓ Setup complete!")

**Reading the output:**

The `Setup complete!` message confirms that all imports loaded. This notebook introduces the classification toolkit the Health Department project demands: `LogisticRegression` for probability-based diagnosis, `DummyClassifier` for the "predict everyone benign" baseline, and `confusion_matrix` plus `classification_report` for dissecting exactly which patients the model gets wrong. We also import the breast cancer dataset itself (`load_breast_cancer`) and a synthetic data generator (`make_classification`) for later experiments. The usual display settings and **RANDOM_SEED = 474** remain in effect.

**Why this matters:** Regression tools (MAE, RMSE, R-squared) are useless when the target is malignant vs. benign. Classification requires its own metrics — accuracy, log loss, confusion matrices — and its own model family. Recognizing which toolkit matches which problem type is the first step toward building the screening tool.

---

## 1. Load Classification Dataset

The Health Department's screening project uses the **Breast Cancer Wisconsin** dataset, a benchmark included in scikit-learn that mirrors the real clinical workflow: a radiologist performs a fine-needle aspirate, a lab digitizes the cell image, and software extracts **30 numeric features** describing each nucleus — `mean_radius`, `mean_texture`, `worst_concave_points`, and so on. Each of the **569 samples** is labeled **0 = malignant** or **1 = benign** based on pathology confirmation.

This is the dataset the MedScreen tool will learn from. Stratified splits preserve the original class balance (~37% malignant, ~63% benign) in every partition, so no fold accidentally under-represents the cancer cases the oncologist cares about most.

> 💡 **Gemini Prompt:** "Load the breast cancer dataset from sklearn with as_frame=True. Print the dataset shape, target class names, class distribution counts, and normalized class balance. Then split into 60/20/20 train/val/test using stratified splitting with random_state=RANDOM_SEED."
>
> **After running, verify:**
> - Dataset has 569 samples and 30 features
> - Two classes: malignant (0) and benign (1) with their counts shown
> - Train/Val/Test sizes printed, preserving class proportions via stratify> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Load breast cancer dataset (binary classification)
data = load_breast_cancer(as_frame=True)
df = data.frame
X = data.data
y = data.target

print(f"Dataset: {data.DESCR.split('===')[0].strip()}")
print(f"\nShape: {X.shape}")
print(f"Target classes: {data.target_names}")
print(f"Class distribution:")
print(y.value_counts())
print(f"\nClass balance: {y.value_counts(normalize=True).round(3).to_dict()}")

# Split data
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_temp)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)} (locked)")

**Reading the output:**

The output confirms **569 samples** and **30 features** — every measurement a lab can extract from a digitized cell nucleus. The two target classes are `malignant` (0) and `benign` (1), distributed roughly **212 malignant** and **357 benign** — a **37%/63%** split. This mild imbalance matters: a model that blindly stamps every aspirate "benign" would already hit ~63% accuracy, sending 212 cancer patients home undiagnosed. Raw accuracy alone cannot be trusted as a performance indicator for the screening tool.

The stratified splits produce approximately **341 training**, **114 validation**, and **114 test** samples, each preserving the 37/63 class ratio. The test set is locked away to simulate the moment MedScreen encounters a brand-new patient at a partner hospital.

**Key takeaway:** Always inspect class balance before modeling. When one class dominates, accuracy inflates and you need additional metrics — precision, recall, confusion matrix — to assess whether the tool actually catches cancer.

---

## 2. Classification Baselines

Before the Health Department invests in model development, the board will ask: *"How much better is your classifier than doing nothing?"* A baseline answers that question by establishing the performance floor — the score any real model must beat to justify its existence.

**Common baselines for screening:**
- **Most frequent class**: Always predict benign (the majority class) — equivalent to a clinic that sends every patient home without testing
- **Stratified random**: Predict malignant/benign in proportion to training prevalence — no better than a weighted coin flip
- **Domain heuristic**: A simple rule like "flag if `worst_radius` > 17" — quick but ignores the other 29 features

**Key insight**: With 63% of samples benign, a model that *never diagnoses cancer* still achieves 63% accuracy. That number is meaningless for patient outcomes.

> 💡 **Gemini Prompt:** "Create a DummyClassifier baseline using the most_frequent strategy. Fit it on training data, predict on validation, and calculate accuracy. Print which class it always predicts and note why accuracy alone can be misleading."
>
> **After running, verify:**
> - Baseline accuracy reflects the majority class proportion (~63%)
> - The classifier always predicts the same class (benign)
> - Warning about accuracy being misleading is displayed> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Most frequent class baseline
baseline_mf = DummyClassifier(strategy='most_frequent')
baseline_mf.fit(X_train, y_train)

y_pred_baseline = baseline_mf.predict(X_val)
baseline_acc = accuracy_score(y_val, y_pred_baseline)

print("=== BASELINE: MOST FREQUENT CLASS ===")
print(f"Validation Accuracy: {baseline_acc:.4f}")
print(f"\nThis baseline always predicts: {data.target_names[int(baseline_mf.predict([X_train.iloc[0]])[0])]}")
print(f"\n⚠️ Accuracy can be misleading! We need better metrics.")

**Reading the output:**

The `DummyClassifier(strategy='most_frequent')` always predicts benign — the equivalent of a screening program that waves every patient through without examination. Its validation accuracy is approximately **0.63**, which is simply the proportion of benign samples in the validation set. The warning below the score highlights the danger: this "model" has learned *nothing* about cell morphology and would miss every single malignant case. Every cancer patient would leave the clinic undiagnosed.

**Why this matters:** This ~63% accuracy is the floor the MedScreen tool must clear. Any real classifier has to beat it convincingly, and — more importantly — has to beat it *on the malignant class*, not just overall. The next sections build logistic regression to do exactly that.

---

## 3. Logistic Regression: From Log-Odds to Probabilities

A clinician does not want a binary "malignant" or "benign" stamp — they want to know *how likely* it is that a tumor is malignant, so they can decide whether to order an immediate biopsy or schedule routine follow-up. Logistic regression provides exactly this: a calibrated probability between 0 and 1.

### The Math (Simplified)

**Linear combination** (same idea as linear regression, but now applied to cell measurements):
```
z = β₀ + β₁·worst_radius + β₂·mean_texture + ... + β₃₀·worst_symmetry
```

**Logistic function (sigmoid):**
```
P(benign | X) = 1 / (1 + e^(-z))
```

**Properties:**
- Output is always between 0 and 1 (a valid probability the oncologist can act on)
- z = 0 → P = 0.5 (decision boundary — maximum clinical uncertainty)
- Large positive z → P ≈ 1 (strong evidence for benign)
- Large negative z → P ≈ 0 (strong evidence for malignant)

> 💡 **Gemini Prompt:** "Plot the sigmoid (logistic) function over z values from -10 to 10. Add a horizontal dashed red line at y=0.5 for the default threshold and a vertical dashed green line at z=0 for the decision boundary. Label axes and add a legend."
>
> **After running, verify:**
> - Smooth S-shaped curve from 0 to 1 is displayed
> - Red dashed line at y=0.5 marks the default classification threshold
> - Green dashed line at z=0 marks where the sigmoid equals exactly 0.5> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Visualize sigmoid function
z = np.linspace(-10, 10, 200)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(10, 6))
plt.plot(z, sigmoid, linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', label='Default threshold (0.5)')
plt.axvline(x=0, color='g', linestyle='--', alpha=0.5, label='Decision boundary (z=0)')
plt.xlabel('Linear Combination (z)')
plt.ylabel('Probability P(y=1|X)')
plt.title('Logistic (Sigmoid) Function')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("💡 The sigmoid squashes any real number into [0, 1]")
print("💡 Default: if P > 0.5, predict class 1; else predict class 0")

**Reading the output:**

The plot shows the classic **S-shaped sigmoid curve** mapping any real-valued linear combination z to a probability between 0 and 1. The red dashed line at **P = 0.5** marks the default decision threshold, and the green dashed line at **z = 0** marks the corresponding input value. When z is large and positive, the sigmoid saturates near 1 (high confidence the tumor is benign); when z is large and negative, it saturates near 0 (high confidence the tumor is malignant). The transition region around z = 0 is where the model is most uncertain — precisely the cases a clinician should review personally.

**Key takeaway:** The sigmoid is what makes logistic regression a *probability* model rather than just a classifier. For the Health Department's screening tool, this distinction is critical: a prediction of P(benign) = 0.51 warrants urgent biopsy review, while P(benign) = 0.99 supports confident routine follow-up. The raw probability gives oncologists actionable clinical information, not just a binary label.

---

## 4. Logistic Regression Pipeline

With 30 features spanning vastly different scales — `mean_area` ranges in the hundreds while `mean_smoothness` hovers near 0.1 — feeding raw measurements into logistic regression would let large-scale features dominate the coefficients and slow convergence. The pipeline below chains `StandardScaler` (zero-mean, unit-variance normalization) with `LogisticRegression` so that `worst_radius` and `mean_smoothness` contribute on equal footing.

We evaluate with both **accuracy** (fraction of correct diagnoses) and **log loss** (a probability-quality metric that harshly penalizes a model that is *confidently wrong* — exactly the failure mode the Health Department fears most).

> 💡 **Gemini Prompt:** "Build a logistic regression pipeline with StandardScaler and LogisticRegression (random_state=RANDOM_SEED, max_iter=1000). Fit on training data. Get both hard predictions (.predict) and probability estimates (.predict_proba) for train and validation sets. Print accuracy and log loss for both sets, plus improvement over baseline."
>
> **After running, verify:**
> - Train and validation accuracy are both printed (should be well above baseline)
> - Train and validation log loss are printed (lower is better)
> - Improvement over baseline accuracy is shown as a positive number> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Basic logistic regression pipeline
log_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
])

log_pipeline.fit(X_train, y_train)

# Predictions
y_pred_train = log_pipeline.predict(X_train)
y_pred_val = log_pipeline.predict(X_val)

# Probabilities
y_proba_train = log_pipeline.predict_proba(X_train)
y_proba_val = log_pipeline.predict_proba(X_val)

print("=== LOGISTIC REGRESSION ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Val Accuracy: {accuracy_score(y_val, y_pred_val):.4f}")
print(f"\nTrain Log Loss: {log_loss(y_train, y_proba_train):.4f}")
print(f"Val Log Loss: {log_loss(y_val, y_proba_val):.4f}")

print(f"\n✓ Improvement over baseline: {accuracy_score(y_val, y_pred_val) - baseline_acc:.4f}")

**Reading the output:**

The logistic regression pipeline reports **Train Accuracy** and **Val Accuracy**, both likely above **0.96** — a massive jump from the ~63% "always predict benign" baseline. This means the 30 cell-nucleus features carry genuine diagnostic signal: `worst_concave_points`, `worst_radius`, and their peers allow the model to distinguish malignant from benign tissue far better than chance.

**Log loss** values around **0.08–0.12** indicate well-calibrated probabilities. Log loss penalizes confident mistakes exponentially: a model that assigns P(benign) = 0.95 to a malignant tumor receives a far heavier penalty than one that hedges at 0.55. For MedScreen, this means the probability estimates clinicians see are trustworthy, not just the binary labels. The small gap between train and validation metrics suggests minimal overfitting.

**Why this matters:** The "Improvement over baseline" line quantifies the leap — typically **+0.33 or more** in accuracy. But accuracy alone hides *which* errors remain. Two models can both score 97% while differing dramatically in how many cancers they miss. The next sections peel back that number with thresholds and confusion matrices.

---

## 📝 PAUSE-AND-DO Exercise 1 (5 minutes)

**Task:** Build logistic pipeline and compute validation accuracy + log loss.

The pipeline is already built above. Now:
1. Look at the probabilities for a few samples
2. Understand the difference between `.predict()` and `.predict_proba()`
3. Explain why log loss might be better than accuracy

---

> 💡 **Gemini Prompt:** "Create a DataFrame showing the first 10 validation samples with columns: True_Label, Predicted_Class, Prob_Class_0, Prob_Class_1, and Correct (boolean). Print the table and note that probabilities sum to 1, predicted class equals argmax, and confidence varies."
>
> **After running, verify:**
> - DataFrame has 10 rows with true labels, predictions, and both class probabilities
> - Prob_Class_0 + Prob_Class_1 = 1.0 for every row
> - Correct column shows True/False matches between prediction and ground truth> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance


### YOUR ANALYSIS:

**Question 1: What's the difference between `.predict()` and `.predict_proba()`?**  
[Your answer]

**Question 2: Why might log loss be better than accuracy?**  
[Your answer - hint: think about probability quality]

**Question 3: What does a probability of 0.51 vs 0.99 tell you?**  
[Your answer - both predict class 1, but...]

---

## 5. Thresholding Matters!

Logistic regression outputs a probability, but the final diagnosis depends on a **threshold**: if P(benign) >= threshold, report benign. The default is 0.5, but this is a mathematical convenience, not a clinical decision. Lowering the threshold makes MedScreen more aggressive about flagging potential cancers (higher recall for malignant, more false alarms); raising it makes the tool more conservative (fewer false alarms, but more missed cancers).

The Health Department must decide: *is it worse to send a cancer patient home (false negative) or to order an unnecessary biopsy (false positive)?* The answer determines the threshold. The sweep below tests four values to show how a single number reshapes the entire diagnostic profile.

> 💡 **Gemini Prompt:** "Loop over classification thresholds [0.3, 0.5, 0.7, 0.9]. For each threshold, convert class-1 probabilities to predictions using that threshold, compute accuracy, and count predicted positives vs negatives. Collect results into a DataFrame and print it."
>
> **After running, verify:**
> - Four rows showing Threshold, Accuracy, Predicted_Positive, Predicted_Negative
> - Lower thresholds produce more positive predictions, higher thresholds produce fewer
> - Accuracy varies across thresholds, showing that 0.5 is not always optimal> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Try different thresholds
thresholds = [0.3, 0.5, 0.7, 0.9]
threshold_results = []

for thresh in thresholds:
    y_pred_thresh = (y_proba_val[:, 1] >= thresh).astype(int)
    acc = accuracy_score(y_val, y_pred_thresh)
    cm = confusion_matrix(y_val, y_pred_thresh)
    
    threshold_results.append({
        'Threshold': thresh,
        'Accuracy': acc,
        'Predicted_Positive': y_pred_thresh.sum(),
        'Predicted_Negative': len(y_pred_thresh) - y_pred_thresh.sum()
    })

results_df = pd.DataFrame(threshold_results)
print("=== THRESHOLD SENSITIVITY ===")
print(results_df)

print("\n💡 Key insight: Changing the threshold changes predictions!")
print("💡 Default 0.5 is not always optimal")
print("💡 We'll explore this more in upcoming notebooks")

**Reading the output:**

The threshold sensitivity table shows four rows corresponding to thresholds **0.3, 0.5, 0.7, and 0.9**. As the threshold increases, MedScreen becomes more conservative about labeling a tumor benign: the **Predicted_Positive** count (benign predictions) drops and **Predicted_Negative** (flagged for further review) rises. At a very low threshold (0.3), nearly everything is labeled benign — maximizing comfort for patients who are truly benign but potentially letting malignant cases slip through. At a high threshold (0.9), the tool demands near-certainty before clearing a patient, flagging many borderline cases for biopsy.

Accuracy may actually *decrease* at extreme thresholds because the model starts misclassifying clear benign cases. But accuracy is the wrong lens here. The clinically relevant question is: **how many cancers does each threshold miss?**

**Why this matters:** If missing a malignant tumor means delayed treatment and potential metastasis, the Health Department would lower the threshold to 0.3 — accepting more unnecessary biopsies to catch every cancer. If hospital capacity cannot handle the extra biopsies, a higher threshold trades some missed diagnoses for a manageable workload. This is a *policy* decision made by clinicians and the board, not a statistical one made by data scientists.

---

## 📝 PAUSE-AND-DO Exercise 2 (5 minutes)

**Task:** Change threshold from 0.5 and observe metric shifts.

Already done above. Now answer:

---

### YOUR OBSERVATIONS:

**Observation 1: What happens when you lower the threshold?**  
[Hint: more/fewer positive predictions?]

**Observation 2: What happens when you raise the threshold?**  
[Hint: how does it affect prediction distribution?]

**Observation 3: When might you want a threshold other than 0.5?**  
[Think about business costs]

---

## 6. Regularized Logistic Regression

Scikit-learn's `LogisticRegression` applies L2 regularization by default, controlled by the inverse-regularization parameter **C**. Lower C means stronger penalty — the model shrinks coefficients toward zero, preventing any single feature like `worst_concave_points` from dominating the diagnosis. This is the classification analogue of Ridge regression from the previous notebook.

We sweep C across five orders of magnitude (0.01 to 100) to see how regularization strength affects the screening tool's accuracy. Strong regularization (C=0.01) forces the model to spread its "attention" across many features; weak regularization (C=100) lets it rely heavily on whichever features best separate malignant from benign in the training data — risking overfitting to the specific 341 training patients.

> 💡 **Gemini Prompt:** "Loop over C values [0.01, 0.1, 1.0, 10.0, 100.0] for logistic regression (each in a StandardScaler pipeline). For each C, fit on training data and record train accuracy, validation accuracy, and the gap. Print the comparison table and identify the best C by validation accuracy."
>
> **After running, verify:**
> - Five rows showing C, Train_Acc, Val_Acc, and Gap for each regularization strength
> - Lower C shows stronger regularization (potentially lower train accuracy)
> - Best C value by validation accuracy is printed at the bottom> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Regularized logistic regression (lower C = stronger regularization)
log_reg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=1000))
])

log_reg_pipeline.fit(X_train, y_train)

# Compare different C values
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
reg_results = []

for C in C_values:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(C=C, random_state=RANDOM_SEED, max_iter=1000))
    ])
    pipe.fit(X_train, y_train)
    
    train_acc = pipe.score(X_train, y_train)
    val_acc = pipe.score(X_val, y_val)
    
    reg_results.append({
        'C': C,
        'Train_Acc': train_acc,
        'Val_Acc': val_acc,
        'Gap': train_acc - val_acc
    })

reg_df = pd.DataFrame(reg_results)
print("=== REGULARIZATION STRENGTH (C parameter) ===")
print(reg_df)
print("\n💡 Lower C = stronger regularization")
print("💡 Look for good validation performance without huge train-val gap")

best_C = reg_df.loc[reg_df['Val_Acc'].idxmax(), 'C']
print(f"\n✓ Best validation accuracy at C = {best_C}")

**Reading the output:**

The table shows five rows for **C = 0.01, 0.1, 1.0, 10.0, and 100.0**. At very low C (strong regularization), both train and validation accuracy may dip slightly because the model is too constrained to fully exploit features like `worst_radius` and `mean_concavity`. As C increases (weaker regularization), accuracy improves and then plateaus. The **Gap** column (train minus validation accuracy) should remain small across all values, confirming that logistic regression is not highly prone to overfitting on this 30-feature, 341-sample dataset. The "Best validation accuracy at C = ..." line identifies the sweet spot.

**Key takeaway:** The C parameter controls how aggressively MedScreen relies on its strongest features. In practice, you would use cross-validation (e.g., `LogisticRegressionCV`) to select C automatically — just as we used `RidgeCV` and `LassoCV` for regression. The next notebook's CV framework will formalize this search.

---

## 7. Confusion Matrix

The board will not ask "What is your accuracy?" They will ask: *"How many cancers did we miss?"* A confusion matrix answers this directly by tabulating all four outcomes of MedScreen's predictions: true positives, true negatives, false positives, and false negatives.

In the screening context, each cell has a concrete clinical meaning — and a very different cost. A **false negative** (telling a cancer patient they are healthy) can be catastrophic: delayed treatment, metastasis, a life at risk. A **false positive** (flagging a healthy patient for biopsy) causes anxiety and expense, but the patient survives. The heatmap below makes these counts visible at a glance.

> 💡 **Gemini Prompt:** "Plot a confusion matrix heatmap for the validation set using ConfusionMatrixDisplay with the breast cancer target names as labels and a Blues colormap. Print TN, FP, FN, TP values and explain what false positives and false negatives mean in medical diagnosis context."
>
> **After running, verify:**
> - Confusion matrix heatmap displayed with malignant and benign labels
> - Four values (TN, FP, FN, TP) are printed with clear labels
> - Medical context explains FP = false alarm, FN = missed diagnosis> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Confusion matrix
cm = confusion_matrix(y_val, y_pred_val)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=data.target_names)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Validation Set')
plt.tight_layout()
plt.show()

print("\n=== CONFUSION MATRIX INTERPRETATION ===")
print(f"True Negatives (TN): {cm[0, 0]}")
print(f"False Positives (FP): {cm[0, 1]}")
print(f"False Negatives (FN): {cm[1, 0]}")
print(f"True Positives (TP): {cm[1, 1]}")

print("\n💡 In medical diagnosis:")
print("   FP = False alarm (predicted malignant, actually benign)")
print("   FN = Missed diagnosis (predicted benign, actually malignant)")
print("\n⚠️ Which error is more costly? This drives threshold choice!")

**Reading the output:**

The confusion matrix heatmap is a 2x2 grid with true labels on the y-axis and predicted labels on the x-axis. The diagonal cells show correct diagnoses: **true negatives** (correctly identified malignant — these patients get timely treatment) and **true positives** (correctly identified benign — these patients avoid unnecessary procedures). The off-diagonal cells show errors:

- **False Positives** (top-right): Malignant cases the model mistakenly called benign — these patients might be sent home without treatment. This is the most dangerous error in screening.
- **False Negatives** (bottom-left): Benign cases the model mistakenly flagged as malignant — these patients receive an unnecessary biopsy but ultimately learn they are healthy.

The printed counts below the chart give exact numbers. The board will focus on one number above all others: the **FP count** (using scikit-learn's convention where malignant=0), because each one represents a potential missed cancer.

**Why this matters:** The confusion matrix is the foundation for every metric in the next notebook — precision, recall, F1-score, ROC curves. Learning to read it fluently, and to identify which cell represents the costliest error for MedScreen's deployment, is the most important skill in applied classification.

---

## 8. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **Logistic Regression**: Maps linear combinations to probabilities via sigmoid
2. **Probabilities vs Classes**: `.predict_proba()` gives you more information than `.predict()`
3. **Thresholds Matter**: Default 0.5 is not always optimal
4. **Baselines**: Even naive strategies can have decent accuracy with imbalance
5. **Regularization**: Control C to prevent overfitting

### Critical Rules:

> **"Always look at probabilities, not just classes"**

> **"Accuracy is not enough - confusion matrix reveals errors"**

> **"Thresholds should be tuned to business costs"**

### Next Steps:

- Next notebook: Classification metrics (precision, recall, ROC, PR curves)
- We'll learn how to systematically choose thresholds
- Class imbalance handling strategies

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning with Python* - Classification chapter
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* - Logistic regression foundations
- scikit-learn User Guide: [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- scikit-learn User Guide: [Probability calibration](https://scikit-learn.org/stable/modules/calibration.html)

---



<center>

Thank you!

</center>